# 02 - Bronze Auto Loader Ingestion
## 05 - Bronze Layer Validation

Validates table existence, row counts, schema correctness, and key field nulls for all bronze tables.

In [0]:
%run ../01_setup/00_config

In [0]:
import logging
from pyspark.sql import functions as F
from utils.validation_utils import get_logger, run_table_validation, display_summary

log = get_logger()

## Source file checks

In [0]:
source_paths = {
    bronze_market_prices_table: f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/market_prices",
    bronze_weather_table:       f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/weather",
    bronze_grid_events_table:   f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/grid_events",
}

for table, path in source_paths.items():
    try:
        files = dbutils.fs.ls(path)
        csv_files = [f for f in files if f.name.endswith(".csv")]
        if len(csv_files) == 0:
            raise ValueError(f"[SOURCE] No CSV files found in: {path}")
        log.info(f"[SOURCE] {table} - {len(csv_files)} source file(s) found in {path}")
    except Exception as e:
        raise ValueError(f"[SOURCE] Cannot access source path {path}: {e}")

## Table validation

In [0]:
expected_schemas = {
    bronze_market_prices_table: {
        "required_columns": {
            "event_date", "region", "market_type",
            "price_eur_mwh", "volume_mwh", "source_system", "last_updated_ts"
        },
        "key_columns":    ["event_date", "region", "price_eur_mwh", "volume_mwh"],
        "numeric_checks": []
    },
    bronze_weather_table: {
        "required_columns": {
            "event_date", "region", "temperature_c",
            "wind_speed_kmh", "precipitation_mm", "source_system", "last_updated_ts"
        },
        "key_columns":    ["event_date", "region"],
        "numeric_checks": []
    },
    bronze_grid_events_table: {
        "required_columns": {
            "event_id", "event_date", "region", "asset_id",
            "event_type", "severity", "source_system", "last_updated_ts"
        },
        "key_columns":    ["event_id", "event_date", "region"],
        "numeric_checks": []
    }
}

results, validation_passed = run_table_validation(spark, expected_schemas, log)

## Validation summary

In [0]:
display_summary(spark, results, validation_passed, "Bronze")